# Phase 7 Lab — Reference Solution

**Phase:** Supervised Learning  
**Scenario:** A modelling review compares algorithms for house-price regression and customer-churn classification.

**Deliverable:** Two tuned, explainable modelling pipelines with identical validation protocols and business-facing recommendations.

Use this only after completing your own attempt. Compare design decisions—not merely output.

## Requirements

        1. Build linear/logistic baselines.
2. Compare regularized, tree, forest, boosting, neighbour, and margin-based candidates where appropriate.
3. Use nested or carefully separated tuning and evaluation.
4. Report uncertainty, residual/error slices, and calibration.
5. Use held-out permutation importance or equivalent explanations.
6. Perform sensitivity analysis on important hyperparameters.
7. Create model cards with intended and excluded uses.

        ## Acceptance criteria

        - The notebook runs from a clean kernel in order.
        - Inputs and outputs have explicit contracts.
        - Invalid, missing, extreme, duplicate, and unseen cases are considered.
        - Important invariants use assertions or tests.
        - Results include interpretation and limitations.
        - Generated artifacts are written under the course `artifacts/` folder.

## Planning worksheet

Complete before coding:

| Question | Your answer |
|---|---|
| What decision or user does the result serve? | |
| What does one row/object/event represent? | |
| What are the required inputs and types? | |
| What outputs and side effects are allowed? | |
| Which assumptions are most fragile? | |
| What is the simplest valid baseline? | |
| Which edge cases must be tested? | |
| How will you know the result is correct? | |

In [ ]:
from pathlib import Path
import sys
import json
import warnings
warnings.filterwarnings("ignore")

_candidates = [Path.cwd(), *Path.cwd().parents]
COURSE_ROOT = next((p for p in _candidates if (p / "datasets").exists()), Path.cwd())
DATA_DIR = COURSE_ROOT / "datasets"
ARTIFACT_DIR = COURSE_ROOT / "artifacts"
ARTIFACT_DIR.mkdir(exist_ok=True)
sys.path.insert(0, str(COURSE_ROOT))

import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from IPython.display import display

RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)
print(f"Course root: {COURSE_ROOT}")

## Reference implementation

This is one defensible solution, not the only correct design. Identify at least one improvement before adopting it.

In [ ]:
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestRegressor, HistGradientBoostingClassifier
from sklearn.impute import SimpleImputer
from sklearn.inspection import permutation_importance
from sklearn.linear_model import Ridge, LogisticRegression
from sklearn.metrics import mean_absolute_error, roc_auc_score
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

house=pd.read_csv(DATA_DIR/"house_prices.csv")
X=house.drop(columns="price"); y=house.price
num=X.select_dtypes(include="number").columns; cat=X.select_dtypes(exclude="number").columns
prep=ColumnTransformer([
    ("num",Pipeline([("impute",SimpleImputer(strategy="median")),("scale",StandardScaler())]),num),
    ("cat",Pipeline([("impute",SimpleImputer(strategy="most_frequent")),
                     ("onehot",OneHotEncoder(handle_unknown="ignore"))]),cat),
])
Xtr,Xte,ytr,yte=train_test_split(X,y,test_size=.2,random_state=42)
candidates={
    "ridge":Pipeline([("prep",prep),("model",Ridge(alpha=5))]),
    "forest":Pipeline([("prep",prep),("model",RandomForestRegressor(n_estimators=180,min_samples_leaf=3,random_state=42,n_jobs=1))]),
}
rows=[]
for name,model in candidates.items():
    model.fit(Xtr,ytr)
    rows.append({"model":name,"MAE":mean_absolute_error(yte,model.predict(Xte))})
display(pd.DataFrame(rows).sort_values("MAE"))

churn=pd.read_csv(DATA_DIR/"customer_churn.csv")
X=churn.drop(columns=["customer_id","churn"]); y=churn.churn
num=X.select_dtypes(include="number").columns; cat=X.select_dtypes(exclude="number").columns
prep2=ColumnTransformer([
    ("num",Pipeline([("impute",SimpleImputer(strategy="median")),("scale",StandardScaler())]),num),
    ("cat",Pipeline([("impute",SimpleImputer(strategy="most_frequent")),
                     ("onehot",OneHotEncoder(handle_unknown="ignore"))]),cat),
])
Xtr,Xte,ytr,yte=train_test_split(X,y,test_size=.2,stratify=y,random_state=42)
classifier=Pipeline([("prep",prep2),("model",LogisticRegression(max_iter=1000,class_weight="balanced"))]).fit(Xtr,ytr)
print("Churn holdout ROC-AUC:",roc_auc_score(yte,classifier.predict_proba(Xte)[:,1]))

## Solution review

Review the reference under four lenses:

1. **Correctness:** Are contracts and calculations enforced?
2. **Robustness:** What failures remain unhandled?
3. **Maintainability:** Which responsibilities should become modules/functions?
4. **Decision validity:** Do outputs support the stated use without overclaiming?

Extend the implementation with one additional test and one observability improvement.